<a href="https://colab.research.google.com/github/seirah-yang/libraries.zip/blob/main/PRT_randomURL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from sqlalchemy.orm import declarative_base, sessionmaker
from sqlalchemy import Column, Integer, String, create_engine
import uuid

Base = declarative_base()

class Task(Base):
    __tablename__ = "tasks"

    id = Column(Integer, primary_key=True)
    public_id = Column(String(36), unique=True, nullable=False, default=lambda: str(uuid.uuid4()))
    task_name = Column(String(100), nullable=False)

# Create an in-memory SQLite database engine
engine = create_engine('sqlite:///:memory:')

# Create tables defined in Base
Base.metadata.create_all(engine)

# Create a sessionmaker to produce Session objects
Session = sessionmaker(bind=engine)

# Create a session instance
session = Session()

task = Task(task_name="FC705-3")
print(f"Before INSERT: task.public_id = {task.public_id}") # Should print None

session.add(task)
session.commit()

print(f"After INSERT: task.public_id = {task.public_id}") # Should print a UUID string
print(f"After INSERT: task.id = {task.id}")

Before INSERT: task.public_id = None
After INSERT: task.public_id = 562f4661-a8aa-4ad7-b2a5-39068af4de1e
After INSERT: task.id = 1


In [16]:
from sqlalchemy.orm import declarative_base, sessionmaker
from sqlalchemy import Column, Integer, String, create_engine
import secrets
import string


Base = declarative_base()


def short_id(length=8):
    """대문자 8 자 랜덤 식별자 생성 (보안용 랜덤 사용)"""
    return ''.join(secrets.choice(string.ascii_uppercase) for _ in range(length))


class Task(Base):
    __tablename__ = "tasks"

    id = Column(Integer, primary_key=True)
    public_id = Column(String(8), unique=True, nullable=False, default=lambda: short_id(8))
    task_name = Column(String(100), nullable=False)


# Create an in-memory SQLite database engine
engine = create_engine('sqlite:///:memory:')


# Create tables defined in Base
Base.metadata.create_all(engine)


# Create a sessionmaker to produce Session objects
Session = sessionmaker(bind=engine)


# Create a session instance
session = Session()


task = Task(task_name="FC705-3")
print(f"Before INSERT: task.public_id = {task.public_id}")  # None

session.add(task)
session.commit()

print(f"After INSERT: task.public_id = {task.public_id}")  # 대문자 8 자, 예: XKQBMJTR
print(f"After INSERT: task.id = {task.id}")

Before INSERT: task.public_id = None
After INSERT: task.public_id = EOGARHXC
After INSERT: task.id = 1


In [18]:
from sqlalchemy.orm import declarative_base, sessionmaker
from sqlalchemy import Column, Integer, String, create_engine
import secrets
import string

# 단일 과제 생성용
Base = declarative_base()


def short_id():
    """대문자 3 개 + 숫자 3 개 조합 (예: XKQ394)"""
    letters = ''.join(secrets.choice(string.ascii_uppercase) for _ in range(3))
    digits = ''.join(secrets.choice(string.digits) for _ in range(3))
    return letters + digits


class Task(Base):
    __tablename__ = "tasks"

    id = Column(Integer, primary_key=True)
    public_id = Column(String(6), unique=True, nullable=False, default=lambda: short_id())
    task_name = Column(String(100), nullable=False)


# Create an in-memory SQLite database engine
engine = create_engine('sqlite:///:memory:')


# Create tables defined in Base
Base.metadata.create_all(engine)


# Create a sessionmaker to produce Session objects
Session = sessionmaker(bind=engine)


# Create a session instance
session = Session()


task = Task(task_name="FC705-3")
print(f"Before INSERT: task.public_id = {task.public_id}")  # None

session.add(task)
session.commit()

print(f"After INSERT: task.public_id = {task.public_id}")  # 대문자 3 + 숫자 3, 예: XKQ394
print(f"After INSERT: task.id = {task.id}")

Before INSERT: task.public_id = None
After INSERT: task.public_id = KWB501
After INSERT: task.id = 1


In [22]:
# DB 입력하여 단체 Random 번호 출력용
from sqlalchemy.orm import declarative_base, sessionmaker
from sqlalchemy import Column, Integer, String, create_engine
import secrets
import string
import pandas as pd


Base = declarative_base()


def short_id():
    """대문자 3 개 + 숫자 3 개 조합 (예: XKQ394)"""
    letters = ''.join(secrets.choice(string.ascii_uppercase) for _ in range(3))
    digits = ''.join(secrets.choice(string.digits) for _ in range(3))
    return letters + digits


class Task(Base):
    __tablename__ = "tasks"

    id = Column(Integer, primary_key=True)
    public_id = Column(String(6), unique=True, nullable=False, default=lambda: short_id())
    task_name = Column(String(100), nullable=False)


# Create an in-memory SQLite database engine
engine = create_engine('sqlite:///:memory:')

# Create tables
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
session = Session()


# 엑셀 파일 읽기 (번호 열 이름은 "과제번호"로 가정)
file_path = "/content/address.csv"  # 실제 파일명으로 교체
df = pd.read_csv(file_path)

# 열 이름이 다를 경우 수정 (예: "과제번호" -> "task_name")
# df.columns = ["task_name", "public_id"]  # 필요시

results = []

for _, row in df.iterrows():
    task_name = row["protocol_num"]  # 엑셀 열 이름에 맞게 수정

    # DB 에 저장
    task = Task(task_name=task_name)
    session.add(task)
    session.commit()

    results.append({
        "과제번호": task_name,
        "변환된값": task.public_id
    })

# 결과 엑셀로 저장
result_df = pd.DataFrame(results)
result_df.to_excel("tasks_converted.xlsx", index=False)

print("변환 완료: tasks_converted.xlsx")
print(result_df)

변환 완료: tasks_converted.xlsx
                    과제번호    변환된값
0                 SJN301  DNE461
1         EU-CTS103-I-01  FBQ308
2                   TEST  AAT945
3              NCCKN-101  XJG584
4             PMC403-A01  ZYE889
..                   ...     ...
101  250509 TEST PROJECT  ZSL496
102             HUC3-637  CUE008
103         HUC1-394-201  DTF505
104              FC705-3  HFA273
105          JPI-547-103  RVR339

[106 rows x 2 columns]
